In [ ]:
%%writefile reg_setup.py
%pip install langchain_community langchainhub chromadb langchain sentence-transformers transformers


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
%pip install -U langchain-huggingface


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain import hub
from langchain_community.llms import HuggingFacePipeline
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings

import torch
from transformers import pipeline

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [ ]:
%pip install ipywidgets hf_xet accelerate

import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# os.environ["LANGCHAIN_API_KEY"] = ""
# os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGCHAIN_ENDPOINT"] = ""
os.environ["LANGCHAIN_API_KEY"] = ""

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
loader = WebBaseLoader(web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"])
docs = loader.load()


In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
splits = text_splitter.split_documents(docs)


In [5]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)
retriever = vectorstore.as_retriever()

print("Total documents in vectorstore:", vectorstore._collection.count())

# Retrieve a sample document
docs = vectorstore.similarity_search("sample query", k=1)
for i, doc in enumerate(docs, 5):
    print(f"\n--- Document {i} ---")
    print("Content:", doc.page_content[:500], "...")  # first 300 chars
    print("Metadata:", doc.metadata)

Total documents in vectorstore: 86

--- Document 5 ---
Content: Memory stream: is a long-term memory module (external database) that records a comprehensive list of agents’ experience in natural language.

Each element is an observation, an event directly provided by the agent.
- Inter-agent communication can trigger new natural language statements.


Retrieval model: surfaces the context to inform the agent’s behavior, according to relevance, recency and importance.

Recency: recent events have higher scores
Importance: distinguish mundane from core memorie ...
Metadata: {'title': "LLM Powered Autonomous Agents | Lil'Log", 'description': 'Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.\n

In [6]:
# HuggingFace LLM pipeline
hf_model = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    device=-1,  # CPU
    max_new_tokens=700,
    # temperature=0.7
)
llm = HuggingFacePipeline(pipeline=hf_model)

Device set to use cpu
C:\Users\Ishita\AppData\Local\Temp\ipykernel_17540\547131069.py:9: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_model)


In [10]:
# Prompt from LangChain Hub
# prompt = hub.pull("rlm/rag-prompt")

from langchain.prompts import PromptTemplate

prompt_template = """
Use the following context to answer the question.
If you don't know the answer, say "I don't know."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)


In [8]:
def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

In [11]:
# RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [12]:
response = rag_chain.invoke("What’s the main idea behind using LLMs as autonomous agents?")
print(response)

scientific discovery, to handle autonomous design, planning, and performance of complex scientific experiments.


In [ ]:
# response = rag_chain.invoke("Why do LLM agents need external tools, and what are some examples?")
# print(response)

extra information that is missing from the model weights (often hard to change after pre-training), including current information, code execution capability, access to proprietary information sources and more.


In [ ]:
# response = rag_chain.invoke("What is the short-term memory and long term memory?")
# print(response)

Token indices sequence length is longer than the specified maximum sequence length for this model (602 > 512). Running this sequence through the model will result in indexing errors


It stores information that we are currently aware of and needed to carry out complex cognitive tasks such as learning and reasoning. Short-term memory is believed to have the capacity of about 7 items (Miller 1956) and lasts for 20-30 seconds. Long-Term Memory (LTM): Long-term memory can store information for a remarkably long time, ranging from a few days to decades, with an essentially unlimited storage capacity.


In [ ]:
# print(f"Vectorstore contains {len(vectorstore.get()['ids'])} documents")


Vectorstore contains 408 documents


In [13]:
def ask_question(question: str) -> str:
    return rag_chain.invoke(question)